# PROYECTO CAPSTONE

## Análisis y modelado predictivo del margen de comercialización

Este notebook desarrolla el proceso analítico del PROYECTO:

1. Carga de datos originales.
2. Limpieza y preparación de información.
3. Construcción de la base mensual estación–mes–combustible.
4. Análisis exploratorio.
5. Creación de variables derivadas.
6. Benchmarking de modelos históricos, estadísticos y de machine learning.
7. Conexión entre revisión de literatura, modelos evaluados y resultados.
8. Evaluación de resultados.
9. Selección del modelo candidato principal.

In [ ]:
# ==============================
# 1. CARGA DE LIBRERÍAS
# ==============================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:,.4f}".format)

print("Librerías cargadas correctamente.")

In [ ]:
# ==============================
# 2. SUBIR ARCHIVO EXCEL
# ==============================

from google.colab import files

uploaded = files.upload()

print("Archivo cargado correctamente:")
for nombre_archivo in uploaded.keys():
    print(nombre_archivo)

In [ ]:
# ==============================
# 3. VER ARCHIVOS DISPONIBLES
# ==============================

import os

os.listdir()

In [ ]:
# ==============================
# 4. LECTURA DE HOJAS DEL EXCEL
# ==============================

archivo = list(uploaded.keys())[0]

excel = pd.ExcelFile(archivo)

print("Archivo utilizado:", archivo)
print("Hojas encontradas en el archivo:")
print(excel.sheet_names)

In [ ]:
# ==============================
# 5. CARGA DE CADA HOJA
# ==============================

volumen = pd.read_excel(archivo, sheet_name="VOLUMEN")
margenes = pd.read_excel(archivo, sheet_name="MARGENES")
pvp = pd.read_excel(archivo, sheet_name="PRECIO VENTA AL PUBLICO")
terminal = pd.read_excel(archivo, sheet_name="PRECIO TERMINAL")

print("Volumen:", volumen.shape)
print("Márgenes:", margenes.shape)
print("PVP:", pvp.shape)
print("Precio terminal:", terminal.shape)

In [ ]:
print("VOLUMEN")
display(volumen.head())

print("MÁRGENES")
display(margenes.head())

print("PVP")
display(pvp.head())

print("PRECIO TERMINAL")
display(terminal.head())

In [ ]:
# ==============================
# 6. REVISIÓN DE COLUMNAS
# ==============================

print("COLUMNAS DE VOLUMEN:")
print(volumen.columns)

print("\nCOLUMNAS DE MÁRGENES:")
print(margenes.columns)

print("\nCOLUMNAS DE PVP:")
print(pvp.columns)

print("\nCOLUMNAS DE PRECIO TERMINAL:")
print(terminal.columns)

In [ ]:
# ==============================
# 7. VISTA PREVIA DE LAS BASES
# ==============================

print("VOLUMEN")
display(volumen.head())

print("MÁRGENES")
display(margenes.head())

print("PVP")
display(pvp.head())

print("PRECIO TERMINAL")
display(terminal.head())

In [ ]:
print("Volumen:", volumen.shape)
print("Márgenes:", margenes.shape)
print("PVP:", pvp.shape)
print("Precio terminal:", terminal.shape)
print(volumen.columns)
print(margenes.columns)
print(pvp.columns)
print(terminal.columns)

In [ ]:
# ==============================
# 8. LIMPIEZA DE NOMBRES DE COLUMNAS
# ==============================

import unicodedata
import re

def limpiar_nombre_columna(columna):
    columna = str(columna).strip().lower()
    columna = unicodedata.normalize("NFKD", columna).encode("ascii", "ignore").decode("utf-8")
    columna = re.sub(r"\s+", "_", columna)
    columna = re.sub(r"[^a-z0-9_]", "", columna)
    return columna

def limpiar_columnas(df):
    df = df.copy()
    df.columns = [limpiar_nombre_columna(col) for col in df.columns]
    return df

volumen = limpiar_columnas(volumen)
margenes = limpiar_columnas(margenes)
pvp = limpiar_columnas(pvp)
terminal = limpiar_columnas(terminal)

print("Columnas limpias de VOLUMEN:")
print(volumen.columns)

print("\nColumnas limpias de MÁRGENES:")
print(margenes.columns)

print("\nColumnas limpias de PVP:")
print(pvp.columns)

print("\nColumnas limpias de PRECIO TERMINAL:")
print(terminal.columns)

In [ ]:
# ==============================
# 9. MAPA DE MESES
# ==============================

mapa_meses = {
    "ENERO": 1,
    "FEBRERO": 2,
    "MARZO": 3,
    "ABRIL": 4,
    "MAYO": 5,
    "JUNIO": 6,
    "JULIO": 7,
    "AGOSTO": 8,
    "SEPTIEMBRE": 9,
    "SETIEMBRE": 9,
    "OCTUBRE": 10,
    "NOVIEMBRE": 11,
    "DICIEMBRE": 12
}

print("Mapa de meses creado correctamente.")

In [ ]:
# ==============================
# 10. PREPARACIÓN DE LA BASE DE VOLUMEN
# ==============================

volumen = volumen.copy()

# Renombramos año para manejarlo sin tilde
volumen = volumen.rename(columns={"ano": "anio"})

# Limpieza de campos clave
volumen["cod_arch"] = volumen["cod_arch"].astype(str).str.strip()
volumen["producto"] = volumen["producto"].astype(str).str.strip().str.lower()
volumen["distribuidor"] = volumen["distribuidor"].astype(str).str.strip()

# Crear número de mes
volumen["mes_num"] = volumen["mes"].astype(str).str.strip().str.upper().map(mapa_meses)

# Convertir galones a número
volumen["galones_original"] = volumen["galones"]
volumen["galones"] = pd.to_numeric(volumen["galones"], errors="coerce")

# Revisar si hay galones no numéricos
galones_no_numericos = volumen[volumen["galones"].isna()]

print("Registros con galones no numéricos:", galones_no_numericos.shape[0])
display(galones_no_numericos)

# Corrección: si aparece algún valor no numérico, lo dejamos en 0 para poder consolidar la base
volumen["galones"] = volumen["galones"].fillna(0)

# Crear fecha mensual
volumen["fecha"] = pd.to_datetime(
    volumen["anio"].astype(str) + "-" + volumen["mes_num"].astype(str) + "-01"
)

print("Base de volumen preparada:")
print(volumen.shape)

display(volumen.head())

In [ ]:
# ==============================
# 11. AGREGACIÓN A NIVEL ESTACIÓN - MES - COMBUSTIBLE
# ==============================

volumen_mensual = (
    volumen
    .groupby(
        ["anio", "mes_num", "fecha", "cod_arch", "producto"],
        as_index=False
    )
    .agg({
        "galones": "sum",
        "distribuidor": "first"
    })
)

print("Registros originales de volumen:", volumen.shape[0])
print("Observaciones mensuales consolidadas:", volumen_mensual.shape[0])

display(volumen_mensual.head())

In [ ]:
# ==============================
# 12. PREPARACIÓN DE LA BASE DE MÁRGENES
# ==============================

margenes = margenes.copy()

margenes["cod_arch"] = margenes["cod_arch"].astype(str).str.strip()

margenes_long = margenes.melt(
    id_vars=["cod_arch", "nombre_estacion_de_servicios"],
    value_vars=["diesel", "extra", "super"],
    var_name="producto",
    value_name="margen_pct"
)

margenes_long["producto"] = margenes_long["producto"].astype(str).str.strip().str.lower()
margenes_long["margen_pct"] = pd.to_numeric(margenes_long["margen_pct"], errors="coerce")

print("Base de márgenes en formato largo:")
print(margenes_long.shape)

display(margenes_long.head())

In [ ]:
# ==============================
# 13. CORRECCIÓN DE ESCALA EN MÁRGENES
# ==============================

margenes_mayores_5 = margenes_long[margenes_long["margen_pct"] > 0.05]

print("Registros de margen mayores al 5% antes de corregir:", margenes_mayores_5.shape[0])
display(margenes_mayores_5.head(20))

# Corrección de escala: 0.9200 pasa a 0.0092
margenes_long.loc[margenes_long["margen_pct"] > 0.05, "margen_pct"] = (
    margenes_long.loc[margenes_long["margen_pct"] > 0.05, "margen_pct"] / 100
)

print("Máximo margen después de corregir:")
print(margenes_long["margen_pct"].max())

In [ ]:
# ==============================
# 14. PREPARACIÓN DEL PVP
# ==============================

pvp = pvp.copy()
pvp = pvp.rename(columns={"ano": "anio"})

pvp_long = pvp.melt(
    id_vars=["fecha", "anio", "mes", "mes_num", "iva"],
    value_vars=["diesel", "extra", "super"],
    var_name="producto",
    value_name="pvp"
)

pvp_long["producto"] = pvp_long["producto"].astype(str).str.strip().str.lower()
pvp_long["pvp"] = pd.to_numeric(pvp_long["pvp"], errors="coerce")

print("Base PVP en formato largo:")
print(pvp_long.shape)

display(pvp_long.head())

In [ ]:
# ==============================
# 15. PREPARACIÓN DEL PRECIO TERMINAL
# ==============================

terminal = terminal.copy()
terminal = terminal.rename(columns={"ano": "anio"})

terminal["mes_num"] = terminal["mes"].astype(str).str.strip().str.upper().map(mapa_meses)

terminal_long = terminal.melt(
    id_vars=["fecha", "anio", "mes", "mes_num", "iva"],
    value_vars=["diesel", "extra", "super"],
    var_name="producto",
    value_name="precio_terminal"
)

terminal_long["producto"] = terminal_long["producto"].astype(str).str.strip().str.lower()
terminal_long["precio_terminal"] = pd.to_numeric(terminal_long["precio_terminal"], errors="coerce")

print("Base precio terminal en formato largo:")
print(terminal_long.shape)

display(terminal_long.head())

In [ ]:
# ==============================
# 16. INTEGRACIÓN DE LA BASE FINAL
# ==============================

base = volumen_mensual.merge(
    margenes_long[["cod_arch", "nombre_estacion_de_servicios", "producto", "margen_pct"]],
    on=["cod_arch", "producto"],
    how="left"
)

base = base.merge(
    pvp_long[["anio", "mes_num", "producto", "pvp"]],
    on=["anio", "mes_num", "producto"],
    how="left"
)

base = base.merge(
    terminal_long[["anio", "mes_num", "producto", "precio_terminal"]],
    on=["anio", "mes_num", "producto"],
    how="left"
)

print("Base integrada:")
print(base.shape)

print("\nValores faltantes por columna:")
print(base.isna().sum())

display(base.head())

In [ ]:
# ==============================
# 17. CREACIÓN DE VARIABLES DEL PROYECTO
# ==============================

base["brecha_precio"] = base["pvp"] - base["precio_terminal"]

base["margen_total_estimado"] = (
    base["galones"] * base["margen_pct"] * base["pvp"]
)

base = base.sort_values(["cod_arch", "producto", "fecha"])

print("Base final con variables creadas:")
print(base.shape)

display(base.head())

In [ ]:
# ==============================
# 18. VALIDACIÓN FINAL
# ==============================

print("Filas y columnas de la base final:", base.shape)

print("\nEstaciones únicas:")
print(base["cod_arch"].nunique())

print("\nProductos:")
print(base["producto"].value_counts())

print("\nPeriodo:")
print(base["fecha"].min(), "a", base["fecha"].max())

print("\nValores faltantes:")
print(base.isna().sum())

print("\nResumen de variables numéricas:")
display(base[["galones", "margen_pct", "pvp", "precio_terminal", "brecha_precio", "margen_total_estimado"]].describe())

In [ ]:
print("Observaciones mensuales consolidadas:", volumen_mensual.shape[0])
print("Base integrada:", base.shape)
print(base.isna().sum())
print(base[["galones", "margen_pct", "pvp", "precio_terminal", "brecha_precio", "margen_total_estimado"]].describe())

In [ ]:
# ==============================
# 19. REVISIÓN DE BRECHA DE PRECIO NEGATIVA
# ==============================

brecha_negativa = base[base["brecha_precio"] < 0].copy()

print("Registros con brecha de precio negativa:", brecha_negativa.shape[0])
print("Porcentaje sobre la base total:", round(brecha_negativa.shape[0] / base.shape[0] * 100, 2), "%")

display(
    brecha_negativa[
        ["anio", "mes_num", "fecha", "cod_arch", "producto", "galones", "pvp", "precio_terminal", "brecha_precio"]
    ].head(20)
)

print("Resumen por producto:")
display(
    brecha_negativa
    .groupby("producto")
    .agg(
        registros=("producto", "count"),
        galones=("galones", "sum"),
        brecha_minima=("brecha_precio", "min"),
        brecha_promedio=("brecha_precio", "mean")
    )
    .reset_index()
)

In [ ]:
# ==============================
# 20. COMPOSICIÓN POR COMBUSTIBLE
# ==============================

composicion_producto = (
    base
    .groupby("producto")
    .agg(
        galones_totales=("galones", "sum"),
        margen_total_estimado=("margen_total_estimado", "sum"),
        observaciones=("producto", "count")
    )
    .reset_index()
)

composicion_producto["pct_volumen"] = (
    composicion_producto["galones_totales"] / composicion_producto["galones_totales"].sum() * 100
)

composicion_producto["pct_margen"] = (
    composicion_producto["margen_total_estimado"] / composicion_producto["margen_total_estimado"].sum() * 100
)

composicion_producto = composicion_producto.sort_values("galones_totales", ascending=False)

display(composicion_producto)

In [ ]:
# ==============================
# 21. GRÁFICO DE COMPOSICIÓN POR COMBUSTIBLE
# ==============================

plt.figure(figsize=(8, 4))
plt.bar(composicion_producto["producto"], composicion_producto["pct_volumen"])
plt.title("Participación del volumen por combustible")
plt.xlabel("Producto")
plt.ylabel("% del volumen total")
plt.grid(axis="y", alpha=0.3)
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(composicion_producto["producto"], composicion_producto["pct_margen"])
plt.title("Participación del margen estimado por combustible")
plt.xlabel("Producto")
plt.ylabel("% del margen total estimado")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# ==============================
# 22. ANÁLISIS ANUAL
# ==============================

analisis_anual = (
    base
    .groupby("anio")
    .agg(
        galones_totales=("galones", "sum"),
        margen_total_estimado=("margen_total_estimado", "sum"),
        margen_promedio=("margen_total_estimado", "mean"),
        observaciones=("producto", "count")
    )
    .reset_index()
)

display(analisis_anual)

plt.figure(figsize=(8, 4))
plt.plot(analisis_anual["anio"], analisis_anual["margen_total_estimado"], marker="o")
plt.title("Evolución anual del margen total estimado")
plt.xlabel("Año")
plt.ylabel("Margen total estimado")
# Configuramos los ticks del eje X para que sean exactamente los años del dataframe
plt.xticks(ticks=analisis_anual["anio"], labels=analisis_anual["anio"].astype(str))
plt.grid(alpha=0.3)
plt.show()

In [ ]:
# ==============================
# 23. ANÁLISIS MENSUAL DEL MARGEN
# ==============================

analisis_mensual = (
    base
    .groupby("fecha")
    .agg(
        galones_totales=("galones", "sum"),
        margen_total_estimado=("margen_total_estimado", "sum")
    )
    .reset_index()
)

display(analisis_mensual.head())

plt.figure(figsize=(12, 4))
plt.plot(analisis_mensual["fecha"], analisis_mensual["margen_total_estimado"], marker="o")
plt.title("Evolución mensual del margen total estimado")
plt.xlabel("Fecha")
plt.ylabel("Margen total estimado")
plt.grid(alpha=0.3)
plt.xticks(rotation=45)
plt.show()

In [ ]:
# ==============================
# 24. MATRIZ DE CORRELACIÓN
# ==============================

variables_correlacion = [
    "galones",
    "margen_pct",
    "pvp",
    "precio_terminal",
    "brecha_precio",
    "margen_total_estimado"
]

correlacion = base[variables_correlacion].corr()

display(correlacion)

plt.figure(figsize=(8, 6))
plt.imshow(correlacion, aspect="auto")
plt.colorbar()
plt.xticks(range(len(variables_correlacion)), variables_correlacion, rotation=45, ha="right")
plt.yticks(range(len(variables_correlacion)), variables_correlacion)
plt.title("Matriz de correlación de variables numéricas")
plt.tight_layout()
plt.show()

In [ ]:
# ==============================
# 25. CREACIÓN DE VARIABLES DE REZAGO
# ==============================

base = base.sort_values(["cod_arch", "producto", "fecha"]).copy()

base["lag_target_1"] = base.groupby(["cod_arch", "producto"])["margen_total_estimado"].shift(1)
base["lag_target_2"] = base.groupby(["cod_arch", "producto"])["margen_total_estimado"].shift(2)
base["lag_target_3"] = base.groupby(["cod_arch", "producto"])["margen_total_estimado"].shift(3)

base["lag_galones_1"] = base.groupby(["cod_arch", "producto"])["galones"].shift(1)
base["lag_galones_2"] = base.groupby(["cod_arch", "producto"])["galones"].shift(2)
base["lag_galones_3"] = base.groupby(["cod_arch", "producto"])["galones"].shift(3)

columnas_rezago = [
    "lag_target_1",
    "lag_target_2",
    "lag_target_3",
    "lag_galones_1",
    "lag_galones_2",
    "lag_galones_3"
]

base_modelo = base.dropna(subset=columnas_rezago).copy()

print("Base integrada antes de rezagos:", base.shape)
print("Base final para modelado:", base_modelo.shape)
print("Observaciones eliminadas por rezagos:", base.shape[0] - base_modelo.shape[0])

print("Base antes de rezagos:", base.shape)
print("Base final para modelado:", base_modelo.shape)

display(base_modelo.head())

In [ ]:
# ==============================
# 26. VALIDACIÓN DE BASE PARA MODELO
# ==============================

print("Filas y columnas:", base_modelo.shape)

print("\nValores faltantes:")
print(base_modelo.isna().sum())

print("\nPeriodo de la base de modelo:")
print(base_modelo["fecha"].min(), "a", base_modelo["fecha"].max())

print("\nVariables disponibles:")
print(base_modelo.columns)

In [ ]:
print("Registros con brecha negativa:", brecha_negativa.shape[0])
display(composicion_producto)
display(correlacion)
print("Base final para modelado:", base_modelo.shape)

In [ ]:
# ==============================
# 27. PREPARACIÓN DE VARIABLES PARA MODELADO
# Benchmarking corregido y alineado a la revisión de literatura
# ==============================

variable_objetivo = "margen_total_estimado"

# Se excluyen variables que pueden generar fuga de información.
# No se usa margen_pct porque participa directamente en la construcción de la variable objetivo.
# No se usa galones del mismo mes como predictor principal, porque se busca anticipar el margen
# con información histórica, comercial y de precios.

variables_predictoras = [
    "pvp",
    "precio_terminal",
    "brecha_precio",
    "lag_target_1",
    "lag_target_2",
    "lag_target_3",
    "lag_galones_1",
    "lag_galones_2",
    "lag_galones_3",
    "mes_num",
    "cod_arch",
    "producto"
]

X = base_modelo[variables_predictoras].copy()
y = base_modelo[variable_objetivo].copy()

print("Variables predictoras:", X.shape)
print("Variable objetivo:", y.shape)

display(X.head())
display(y.head())

In [ ]:
# ==============================
# 28. SEPARACIÓN TEMPORAL ENTRENAMIENTO / PRUEBA
# ==============================

base_modelo = base_modelo.sort_values("fecha").copy()

# Entrenamiento: períodos iniciales
# Prueba: períodos posteriores
train = base_modelo["fecha"] < "2025-01-01"
test = base_modelo["fecha"] >= "2025-01-01"

X_train = base_modelo.loc[train, variables_predictoras]
y_train = base_modelo.loc[train, variable_objetivo]

X_test = base_modelo.loc[test, variables_predictoras]
y_test = base_modelo.loc[test, variable_objetivo]

print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

print("Periodo entrenamiento:")
print(base_modelo.loc[train, "fecha"].min(), "a", base_modelo.loc[train, "fecha"].max())

print("Periodo prueba:")
print(base_modelo.loc[test, "fecha"].min(), "a", base_modelo.loc[test, "fecha"].max())

In [ ]:
# ==============================
# 29. PREPROCESAMIENTO PARA MODELOS
# ==============================

from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

variables_categoricas = ["cod_arch", "producto"]
variables_numericas = [
    col for col in variables_predictoras
    if col not in variables_categoricas
]

# Compatibilidad con distintas versiones de Scikit-learn
try:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
except TypeError:
    encoder = OneHotEncoder(handle_unknown="ignore", sparse=False)

preprocesador = ColumnTransformer(
    transformers=[
        ("categoricas", encoder, variables_categoricas),
        ("numericas", "passthrough", variables_numericas)
    ]
)

print("Preprocesador creado correctamente.")
print("Variables categóricas:", variables_categoricas)
print("Variables numéricas:", variables_numericas)

In [ ]:
# ==============================
# 30. FUNCIÓN DE EVALUACIÓN
# ==============================

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def calcular_metricas(y_real, y_predicho):
    y_real = np.array(y_real)
    y_predicho = np.array(y_predicho)

    mae = mean_absolute_error(y_real, y_predicho)
    rmse = np.sqrt(mean_squared_error(y_real, y_predicho))

    # MAPE solo sobre valores positivos para evitar división entre cero
    mascara = y_real > 0
    mape = np.mean(np.abs((y_real[mascara] - y_predicho[mascara]) / y_real[mascara])) * 100

    r2 = r2_score(y_real, y_predicho)

    return mae, rmse, mape, r2

print("Función de métricas creada correctamente.")

In [ ]:
# ==============================
# 31. BENCHMARKS HISTÓRICOS
# ==============================

# Benchmark 1: Naïve temporal
# Predice el margen actual usando el margen del mes anterior.
pred_naive = base_modelo.loc[test, "lag_target_1"]

mae_naive, rmse_naive, mape_naive, r2_naive = calcular_metricas(y_test, pred_naive)


# Benchmark 2: Media móvil de 3 meses
# Predice el margen actual usando el promedio de los últimos 3 meses.
pred_media_movil_3 = (
    base_modelo.loc[test, ["lag_target_1", "lag_target_2", "lag_target_3"]]
    .mean(axis=1)
)

mae_mm3, rmse_mm3, mape_mm3, r2_mm3 = calcular_metricas(y_test, pred_media_movil_3)


print("NAÏVE TEMPORAL")
print("MAE:", round(mae_naive, 2))
print("RMSE:", round(rmse_naive, 2))
print("MAPE:", round(mape_naive, 2), "%")
print("R²:", round(r2_naive, 4))

print("\nMEDIA MÓVIL 3 MESES")
print("MAE:", round(mae_mm3, 2))
print("RMSE:", round(rmse_mm3, 2))
print("MAPE:", round(mape_mm3, 2), "%")
print("R²:", round(r2_mm3, 4))

In [ ]:
# ==============================
# 32. MODELO 1: REGRESIÓN LINEAL
# ==============================

from sklearn.linear_model import LinearRegression

modelo_regresion = Pipeline(
    steps=[
        ("preprocesador", preprocesador),
        ("modelo", LinearRegression())
    ]
)

modelo_regresion.fit(X_train, y_train)

pred_regresion = modelo_regresion.predict(X_test)

mae_reg, rmse_reg, mape_reg, r2_reg = calcular_metricas(y_test, pred_regresion)

print("REGRESIÓN LINEAL")
print("MAE:", round(mae_reg, 2))
print("RMSE:", round(rmse_reg, 2))
print("MAPE:", round(mape_reg, 2), "%")
print("R²:", round(r2_reg, 4))

In [ ]:
# ==============================
# 33. MODELO 2: RANDOM FOREST
# ==============================

from sklearn.ensemble import RandomForestRegressor

modelo_rf = Pipeline(
    steps=[
        ("preprocesador", preprocesador),
        ("modelo", RandomForestRegressor(
            n_estimators=200,
            random_state=42,
            n_jobs=-1,
            max_depth=None
        ))
    ]
)

modelo_rf.fit(X_train, y_train)

pred_rf = modelo_rf.predict(X_test)

mae_rf, rmse_rf, mape_rf, r2_rf = calcular_metricas(y_test, pred_rf)

print("RANDOM FOREST")
print("MAE:", round(mae_rf, 2))
print("RMSE:", round(rmse_rf, 2))
print("MAPE:", round(mape_rf, 2), "%")
print("R²:", round(r2_rf, 4))

In [ ]:
# ==============================
# 34. MODELO 3: XGBOOST
# ==============================

try:
    from xgboost import XGBRegressor
except:
    !pip install xgboost
    from xgboost import XGBRegressor

modelo_xgb = Pipeline(
    steps=[
        ("preprocesador", preprocesador),
        ("modelo", XGBRegressor(
            n_estimators=200,
            learning_rate=0.05,
            max_depth=4,
            subsample=0.9,
            colsample_bytree=0.9,
            objective="reg:squarederror",
            random_state=42,
            n_jobs=-1
        ))
    ]
)

modelo_xgb.fit(X_train, y_train)

pred_xgb = modelo_xgb.predict(X_test)

mae_xgb, rmse_xgb, mape_xgb, r2_xgb = calcular_metricas(y_test, pred_xgb)

print("XGBOOST")
print("MAE:", round(mae_xgb, 2))
print("RMSE:", round(rmse_xgb, 2))
print("MAPE:", round(mape_xgb, 2), "%")
print("R²:", round(r2_xgb, 4))

In [ ]:
# ==============================
# 35. TABLA FINAL DE BENCHMARKING
# ==============================

resultados_benchmarking = pd.DataFrame({
    "Modelo": [
        "Naïve temporal",
        "Media móvil 3 meses",
        "Regresión lineal",
        "Random Forest",
        "XGBoost"
    ],
    "Rol": [
        "Benchmark histórico",
        "Benchmark histórico",
        "Benchmark estadístico",
        "Ensamble ML",
        "Boosting"
    ],
    "MAE": [
        mae_naive,
        mae_mm3,
        mae_reg,
        mae_rf,
        mae_xgb
    ],
    "RMSE": [
        rmse_naive,
        rmse_mm3,
        rmse_reg,
        rmse_rf,
        rmse_xgb
    ],
    "MAPE": [
        mape_naive,
        mape_mm3,
        mape_reg,
        mape_rf,
        mape_xgb
    ],
    "R2": [
        r2_naive,
        r2_mm3,
        r2_reg,
        r2_rf,
        r2_xgb
    ],
    "Lectura": [
        "Margen del mes anterior",
        "Promedio histórico reciente",
        "Referencia interpretable",
        "Modelo de ensamble",
        "Contraste avanzado"
    ]
})

mae_base = resultados_benchmarking.loc[
    resultados_benchmarking["Modelo"] == "Naïve temporal", "MAE"
].values[0]

rmse_base = resultados_benchmarking.loc[
    resultados_benchmarking["Modelo"] == "Naïve temporal", "RMSE"
].values[0]

resultados_benchmarking["Mejora_MAE_vs_Naive"] = (
    (mae_base - resultados_benchmarking["MAE"]) / mae_base * 100
)

resultados_benchmarking["Mejora_RMSE_vs_Naive"] = (
    (rmse_base - resultados_benchmarking["RMSE"]) / rmse_base * 100
)

resultados_benchmarking_redondeado = resultados_benchmarking.copy()

resultados_benchmarking_redondeado["MAE"] = resultados_benchmarking_redondeado["MAE"].round(2)
resultados_benchmarking_redondeado["RMSE"] = resultados_benchmarking_redondeado["RMSE"].round(2)
resultados_benchmarking_redondeado["MAPE"] = resultados_benchmarking_redondeado["MAPE"].round(1)
resultados_benchmarking_redondeado["R2"] = resultados_benchmarking_redondeado["R2"].round(3)
resultados_benchmarking_redondeado["Mejora_MAE_vs_Naive"] = resultados_benchmarking_redondeado["Mejora_MAE_vs_Naive"].round(2)
resultados_benchmarking_redondeado["Mejora_RMSE_vs_Naive"] = resultados_benchmarking_redondeado["Mejora_RMSE_vs_Naive"].round(2)

display(resultados_benchmarking_redondeado)

In [ ]:
# ==============================
# 36. GRÁFICO COMPARATIVO DE MAE Y RMSE
# ==============================

plt.figure(figsize=(9, 4))
plt.bar(resultados_benchmarking_redondeado["Modelo"], resultados_benchmarking_redondeado["MAE"])
plt.title("Comparación de MAE por modelo")
plt.xlabel("Modelo")
plt.ylabel("MAE")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("Figura_comparacion_MAE_modelos.png", dpi=300)
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(resultados_benchmarking_redondeado["Modelo"], resultados_benchmarking_redondeado["RMSE"])
plt.title("Comparación de RMSE por modelo")
plt.xlabel("Modelo")
plt.ylabel("RMSE")
plt.xticks(rotation=30, ha="right")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("Figura_comparacion_RMSE_modelos.png", dpi=300)
plt.show()

In [ ]:
# ==============================
# 37. SELECCIÓN DEL MODELO CANDIDATO FORMAL
# ==============================

# Los benchmarks históricos se usan como referencia mínima.
# La selección del modelo candidato se realiza entre los modelos formales:
# regresión lineal, Random Forest y XGBoost.

modelos_formales = resultados_benchmarking[
    resultados_benchmarking["Modelo"].isin([
        "Regresión lineal",
        "Random Forest",
        "XGBoost"
    ])
].copy()

mejor_modelo_formal = modelos_formales.sort_values(
    by=["RMSE", "MAE"],
    ascending=[True, True]
).iloc[0]

print("Modelo candidato formal según menor RMSE y MAE:")
print(mejor_modelo_formal["Modelo"])

print("\nMétricas:")
print("MAE:", round(mejor_modelo_formal["MAE"], 2))
print("RMSE:", round(mejor_modelo_formal["RMSE"], 2))
print("MAPE:", round(mejor_modelo_formal["MAPE"], 2), "%")
print("R²:", round(mejor_modelo_formal["R2"], 4))

print("\nCriterio:")
print(
    "Los benchmarks históricos sirven como referencia mínima. "
    "La selección del modelo candidato se realiza entre los modelos formales "
    "identificados en la literatura."
)

In [ ]:
# ==============================
# 38. TABLA FINAL PARA PROYECTO
# ==============================

tabla_7_PROYECTO = resultados_benchmarking_redondeado[
    ["Modelo", "Rol", "MAE", "RMSE", "MAPE", "R2", "Lectura"]
].copy()

display(tabla_7_PROYECTO)

In [ ]:
# ==============================
# 39. CONEXIÓN ENTRE LITERATURA Y MODELOS EVALUADOS
# ==============================

conexion_literatura_modelos = pd.DataFrame({
    "Literatura de referencia": [
        "Hyndman & Athanasopoulos",
        "James et al.; Hyndman & Athanasopoulos",
        "Ahamed et al.; Hwang et al.; Géron",
        "Zhang & Wu; Bussmann et al.; Géron"
    ],
    "Solución identificada": [
        "Forecasting base y referencia histórica",
        "Modelo estadístico interpretable",
        "Machine learning / modelos de ensamble",
        "Boosting / datos tabulares"
    ],
    "Modelo aplicado": [
        "Naïve temporal / Media móvil 3 meses",
        "Regresión lineal",
        "Random Forest",
        "XGBoost"
    ],
    "Resultado del PROYECTO": [
        "Referencias mínimas para medir valor agregado",
        "Benchmark estadístico interpretable",
        "Menor MAE y RMSE entre modelos formales; modelo candidato principal",
        "Contraste avanzado"
    ]
})

display(conexion_literatura_modelos)

In [ ]:
# ==============================
# 40. EXPORTACIÓN DE RESULTADOS PARA TESIS
# ==============================

with pd.ExcelWriter("Resultados_CAPSTONE_corregido.xlsx") as writer:
    composicion_producto.to_excel(writer, sheet_name="Composicion_producto", index=False)
    correlacion.to_excel(writer, sheet_name="Correlacion")
    resultados_benchmarking_redondeado.to_excel(writer, sheet_name="Benchmarking_completo", index=False)
    tabla_7_tesis.to_excel(writer, sheet_name="Tabla_7_benchmarking", index=False)
    conexion_literatura_modelos.to_excel(writer, sheet_name="Tabla_8_literatura_modelos", index=False)
    base.to_excel(writer, sheet_name="Base_integrada", index=False)
    base_modelo.to_excel(writer, sheet_name="Base_modelo", index=False)

print("Archivo exportado correctamente: Resultados_CAPSTONE_corregido.xlsx")

In [ ]:
# ==============================
# 41. RESUMEN EJECUTIVO DEL RESULTADO
# ==============================

print("RESUMEN EJECUTIVO DEL MODELADO")
print("--------------------------------")
print("Modelo candidato formal:", mejor_modelo_formal["Modelo"])
print("MAE:", round(mejor_modelo_formal["MAE"], 2))
print("RMSE:", round(mejor_modelo_formal["RMSE"], 2))
print("MAPE:", round(mejor_modelo_formal["MAPE"], 1), "%")
print("R²:", round(mejor_modelo_formal["R2"], 3))
print("")
print(
    "Interpretación: el modelo candidato se selecciona entre los modelos formales "
    "evaluados, usando como criterio principal RMSE y MAE. Los benchmarks históricos "
    "sirven como referencia mínima para comprobar si los modelos agregan valor frente "
    "al comportamiento histórico del margen."
)